# Three-state entropy for LR, XGB and NN (Fig. 8)

For every lipid and frame the sn-1 and sn-2 labels give one of three states (both Lo, both Ld, mixed). S_3st = −Σ p ln p over the mean populations. Mixed systems: both leaflets; pure DOPC and DOPC/CHOL 60/40: upper leaflet only (the lower leaflet was the training leaflet). Error bar at χ_C = 0.25 only: SD (ddof = 1) over `dopc_75`, `dopc_75_rep1`, `dopc_75_rep2`. Needs the classified labels of all three models.

In [ ]:
import sys
sys.path.insert(0, "..")          # code_libraries is in the repository root; data in ../heavy_data_NO_GIT

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
from matplotlib.container import ErrorbarContainer

heavy_dir = "../heavy_data_NO_GIT"
out_dir = os.path.join(heavy_dir, "results/chain_mix")
os.makedirs(out_dir, exist_ok=True)

# model: (folder with the classified labels, window)
MODELS = {"LR": ("data_arrays_by_chain_logreg", 7), "XGB": ("data_arrays_by_chain_xgb", 17), "NN": ("data_arrays_by_chain_nn", 5)}
# system: (chi_C as simulated, leaflets used: 0 = lower, 1 = upper)
SYSTEMS = {"dopc_pure": (0.0, (1,)), "dopc_94": (0.0625, (0, 1)), "dopc_90": (0.1018, (0, 1)), "dopc_85": (0.1477, (0, 1)),
           "dopc_80": (0.2021, (0, 1)), "dopc_75": (0.25, (0, 1)), "dopc_70": (0.2991, (0, 1)),
           "dopc_65": (0.3506, (0, 1)), "dopc_60": (0.40, (1,))}
REPLICAS = ["dopc_75", "dopc_75_rep1", "dopc_75_rep2"]

In [ ]:
def three_state_entropy(folder, n_point, sys_name, leaflets=(0, 1)):
    """S_3st from the mean fractions of both-Lo, both-Ld and mixed lipids (counted per frame over the chosen leaflets)."""
    lab = {ch: np.load(os.path.join(heavy_dir, folder, f"{sys_name}_{ch}_ft_last_{n_point}pt_prediction_labels.npy"),
                       allow_pickle=True).astype(bool)[list(leaflets)] for ch in ("sn1", "sn2")}
    sn1, sn2 = lab["sn1"], lab["sn2"]
    n_lipids = sn1.shape[0] * sn1.shape[2]
    p = np.array([(sn1 & sn2).sum(axis=(0, 2)).mean(), (~sn1 & ~sn2).sum(axis=(0, 2)).mean(),
                  (sn1 ^ sn2).sum(axis=(0, 2)).mean()]) / n_lipids
    return float(-np.sum([q * np.log(q) for q in p if q > 0]))


ENT = pd.DataFrame({"system": list(SYSTEMS), "chi_C": [v[0] for v in SYSTEMS.values()]})
ENT_SD = {}
for model, (folder, n_point) in MODELS.items():
    ENT[model] = [three_state_entropy(folder, n_point, s, lf) for s, (chi, lf) in SYSTEMS.items()]
    ENT_SD[model] = np.std([three_state_entropy(folder, n_point, s) for s in REPLICAS], ddof=1)
print(ENT.round(4).to_string(index=False))
print("replica SD at chi_C = 0.25: " + ", ".join(f"{k} {v:.4f}" for k, v in ENT_SD.items()))

In [ ]:
# paper figure style: bold text, ticks on all four sides pointing in, minor ticks
PAPER_STYLE = {"font.weight": "bold", "axes.labelweight": "bold", "axes.titleweight": "bold", "axes.linewidth": 2.5,
               "mathtext.default": "bf", "xtick.top": True, "ytick.right": True, "xtick.direction": "in",
               "ytick.direction": "in", "xtick.major.size": 10, "ytick.major.size": 10, "xtick.major.width": 2.5,
               "ytick.major.width": 2.5, "xtick.minor.size": 5, "ytick.minor.size": 5, "xtick.minor.width": 1.8,
               "ytick.minor.width": 1.8, "xtick.major.pad": 8, "ytick.major.pad": 8}
SN = {"sn1": r"$\mathbfit{sn}$-1", "sn2": r"$\mathbfit{sn}$-2"}
LO, LD = r"$\mathbf{L_o}$", r"$\mathbf{L_d}$"


def style_axes(ax, size=20):
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.tick_params(axis="both", which="both", labelsize=size, top=True, right=True, direction="in")
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight("bold")


def legend_without_error_bars(ax, **kwargs):
    """Legend that shows only the line and marker of each curve, not its error bar."""
    handles, labels = ax.get_legend_handles_labels()
    handles = [h[0] if isinstance(h, ErrorbarContainer) else h for h in handles]
    return ax.legend(handles, labels, **kwargs)


ERR = dict(capsize=8, elinewidth=2.5, capthick=2.5, markersize=10)
COLORS = {"LR": "goldenrod", "XGB": "purple", "NN": "teal"}
at_025 = np.isclose(ENT["chi_C"], 0.25)

with plt.rc_context(PAPER_STYLE):
    fig, ax = plt.subplots(figsize=(12, 10))
    ax.axhline(np.log(3), linestyle="dashed", color="black", linewidth=2, label="ln(3) max entropy")
    for model, color in COLORS.items():
        # error bar only at chi_C = 0.25 (where the replicas are); NaN = no error bar
        ax.errorbar(ENT["chi_C"], ENT[model], yerr=np.where(at_025, ENT_SD[model], np.nan), marker="o", color=color,
                    linewidth=3, label=model, **ERR)
    ax.set_xlim(-0.02, 0.42)
    ax.set_ylim(0.7, 1.2)
    ax.set_xlabel("CHOL fraction", fontsize=24)
    ax.set_ylabel(r"$\mathbf{S_{3st}}$ ( nats)", fontsize=24)
    style_axes(ax)
    legend_without_error_bars(ax, loc="lower center", fontsize=22, frameon=True, edgecolor="black")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "entropy_LR_XGB_NN.png"), dpi=200)
    plt.show()